# 07 — GAT Implementation (PyTorch Geometric)
Implement a compact two-layer GAT regressor and verify tensor alignment.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path("..").resolve()
mapping = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "road_segment_mapping.csv")
edges = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "segment_graph_edges.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device)


In [ ]:
try:
    from torch_geometric.nn import GATConv
    TORCH_GEOMETRIC_OK = True
except Exception as e:
    TORCH_GEOMETRIC_OK = False
    print("torch_geometric missing:", e)


In [ ]:
if TORCH_GEOMETRIC_OK:
    class GATRegressor(torch.nn.Module):
        def __init__(self, in_dim=4, hidden_dim=32, heads=2, dropout=0.1):
            super().__init__()
            self.in_proj = torch.nn.Linear(in_dim, hidden_dim)
            self.gat1 = GATConv(hidden_dim, hidden_dim, heads=heads, dropout=dropout)
            self.gat2 = GATConv(hidden_dim*heads, hidden_dim, heads=1, dropout=dropout)
            self.out = torch.nn.Linear(hidden_dim, 1)
            self.drop = torch.nn.Dropout(dropout)

        def forward(self, x, edge_index):
            x = torch.relu(self.in_proj(x))
            x = self.drop(torch.relu(self.gat1(x, edge_index)))
            x = self.drop(torch.relu(self.gat2(x, edge_index)))
            return self.out(x).squeeze(-1)

    N = len(mapping)
    x = torch.randn(N, 4, dtype=torch.float32, device=device)
    edge_index = torch.tensor(edges[["src","dst"]].to_numpy().T, dtype=torch.long, device=device)

    model = GATRegressor().to(device)
    y_hat = model(x, edge_index)
    print("Output shape", tuple(y_hat.shape))
    assert y_hat.shape[0] == N
    assert torch.isfinite(y_hat).all()

    # gradient + optimizer sanity
    y = torch.randn(N, device=device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss = torch.nn.functional.mse_loss(y_hat, y)
    loss.backward()
    opt.step()
    print("optimizer step ok; loss", float(loss))
else:
    print("Install torch-geometric to execute this notebook fully.")


## Edge feature note
Standard `GATConv` here uses topology (`edge_index`) but not custom edge attributes.
If turn attributes are required, switch to an edge-aware operator and document the change.


In [ ]:
print("Node-ID alignment check:")
print(mapping[["road_segment_id","graph_node_id"]].head())


This notebook keeps the first model small and debuggable.
